# ORBIT AI Results Analysis: Domain Shift & Hybrid Inference Engine

This notebook demonstrates the mathematical problem encountered during Cross-Hardware Inference (Covariate Shift) and visualizing the *Washout Effect* of Dynamic Rolling Z-Score Normalization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')

# Simulate the Washout Effect
time_steps = np.arange(0, 100)
# True Focus signal (e.g., continuous math task)
true_signal = np.ones_like(time_steps) * 5.0 + np.random.normal(0, 0.2, len(time_steps))

### Visualizing the Washout Effect

When a user sustains focus for an entire recording (e.g. 60 seconds), a rolling normalization buffer misinterprets the concentration as the *new normal*, reducing the normalized signal back to Zero (IDLE).

In [ ]:
buffer_size = 15
rolling_mean = np.zeros_like(time_steps, dtype=float)
rolling_std = np.zeros_like(time_steps, dtype=float)

# Compute rolling stats
for i in range(len(time_steps)):
    start = max(0, i - buffer_size)
    window = true_signal[start:i+1]
    rolling_mean[i] = np.mean(window)
    rolling_std[i] = np.std(window) + 1e-6
    
normalized_signal = (true_signal - rolling_mean) / rolling_std

plt.figure(figsize=(10, 6))
plt.plot(time_steps, true_signal, label='Raw Focus Signal (High Beta)')
plt.plot(time_steps, rolling_mean, label='Rolling Mean (drifts up)', linestyle='--')
plt.plot(time_steps, normalized_signal, label='Normalized Signal (Washed out to 0)', color='red')
plt.title('The Washout Effect: Continuous Focus Gets Zeroed Out')
plt.xlabel('Time (frames)')
plt.ylabel('Amplitude')
plt.legend()
plt.show()

### Layer 1: True Global Baseline Normalization

Using the static baseline extracted from the 2.5 Million parameter training corpus prevents the washout.

In [ ]:
global_mean = 1.0  # Training baseline IDLE mean
global_std = 0.5   # Expected variation

static_normalized = (true_signal - global_mean) / global_std

plt.figure(figsize=(10, 6))
plt.plot(time_steps, true_signal, label='Raw Focus Signal', alpha=0.5)
plt.plot(time_steps, static_normalized, label='Static Global Normalization (Stays HIGH)', color='green')
plt.axhline(0, color='black', label='IDLE Baseline (0)', linestyle='--')
plt.title('Layer 1 Fix: Static Baseline Preserves Continuous Focus')
plt.xlabel('Time (frames)')
plt.ylabel('Amplitude')
plt.legend()
plt.show()

### Layer 2: Biological Override

Because of physical Covariate Shift between clinical hardware (OpenNeuro) and consumer hardware (TGAM), the model alone can fail. We inject a Biological Filter (Beta > Alpha).

In [ ]:
# Simulating Alpha and Beta powers
alpha_power = np.linspace(-40, -45, 100) + np.random.normal(0, 1, 100)
beta_power = np.linspace(-42, -35, 100) + np.random.normal(0, 1, 100)

attention_feature = beta_power - alpha_power

plt.figure(figsize=(10, 6))
plt.plot(time_steps, alpha_power, label='Alpha Power (Relax)')
plt.plot(time_steps, beta_power, label='Beta Power (Focus)')
plt.fill_between(time_steps, alpha_power, beta_power, where=(beta_power > alpha_power), 
                 color='green', alpha=0.3, label='FORWARD Override Triggered')
plt.title('Layer 2: Biological Feature Override (Beta Dominance)')
plt.xlabel('Time (frames)')
plt.ylabel('dB Power')
plt.legend()
plt.show()